# Project 1 — Issue Report Classification
## Notebook 03: Classical machine learning

**Track B deliverable.**

Four classifier families over a shared TF-IDF representation, plus the two
experiments that decide how the features are built:

1. **Preprocessing ablation** — which cleaning level actually helps.
2. **Hyperparameter search** — cross-validated, on the training split only.
3. **Final results** — the competition protocol, against the SetFit baseline.

The headline finding is in section 1, and it is not the one Track A expected.

In [1]:
import json

import pandas as pd

from ai4se.classical import CLASSICAL_MODELS, TUNED_MODELS, TUNED_PREPROCESSING
from ai4se.evaluation import (
    SETFIT_OVERALL,
    cross_validate,
    leaderboard_from_disk,
    load_result,
)
from ai4se.loader import PROJECT_ROOT, load_split
from ai4se.preprocessing import make_cleaner

pd.set_option("display.width", 140)
TABLES = PROJECT_ROOT / "results" / "tables"

train = load_split("train", kind="memory")
test = load_split("test", kind="memory")
train.apply(make_cleaner(**TUNED_PREPROCESSING))
test.apply(make_cleaner(**TUNED_PREPROCESSING))
print(train, "|", test)
print("preprocessing:", TUNED_PREPROCESSING)

InMemoryIssueRepository(n=1500) | InMemoryIssueRepository(n=1500)
preprocessing: {'level': 'light', 'title_weight': 3, 'max_words': 200}


---
## 1. Does cleaning help? (Preprocessing ablation)

Track A built a three-level cleaning pipeline on the reasonable assumption that
stripping noise from issue bodies would help a bag-of-words model. That
assumption needed testing rather than believing.

Each configuration below was cross-validated with the same logistic-regression
pipeline; only the preprocessing changed.

In [2]:
ablation = pd.DataFrame(json.loads((TABLES / "ablation_preprocessing.json").read_text()))
pivot = ablation.pivot_table(
    index=["level"], columns=["title_weight"], values="macro_f1", aggfunc="max"
)
display(ablation.head(8).reset_index(drop=True))
print("\nbest macro F1 by cleaning level and title weight:")
display(pivot.round(4))

,level,title_weight,max_words,macro_f1,std
0,raw,3,400.0,0.7472,0.0295
1,raw,1,NaN,0.7472,0.0272
2,light,3,200.0,0.7472,0.0287
3,raw,3,200.0,0.7463,0.0314
4,light,1,200.0,0.7461,0.0303
5,light,3,NaN,0.7451,0.0292
6,raw,3,NaN,0.7443,0.0270
7,raw,1,400.0,0.7439,0.0280



best macro F1 by cleaning level and title weight:


title_weight,1,3
level,,
full,0.7303,0.7364
light,0.7461,0.7472
raw,0.7472,0.7472


**`full` cleaning is the worst of the three levels**, by roughly 0.015 macro F1
— about half a standard deviation of the fold-to-fold spread, and consistent
across every title weight and truncation setting.

The explanation is in what `full` removes. Stop-word removal deletes *would*,
*could*, *should* and *please*; lemmatisation collapses tense. Those are
precisely the modal and evaluative words that Track A's own term analysis
identified as the signature of a feature request. The cleaning was removing the
signal along with the noise.

`light` cleaning — which strips code blocks, stack traces, markup and URLs but
leaves natural language intact — is what the rest of Track B uses.

**Two lessons worth stating in the report.** First, a preprocessing step that
is obviously sensible can still be harmful, and only an ablation shows it.
Second, Track A's own analysis contained the warning: code blocks were far more
common in bug reports than in feature requests, which meant stripping them was
never going to be free.

### Title weighting and truncation

In [3]:
display(
    ablation.pivot_table(index="level", columns="max_words", values="macro_f1", aggfunc="max")
    .round(4)
)

max_words,200.0,400.0
level,,
full,0.7356,0.7336
light,0.7472,0.7438
raw,0.7463,0.7472


Repeating the title three times gives a small, consistent gain: titles are
short and dense with exactly the vocabulary that distinguishes the classes.
Truncation past 200 words neither helps nor hurts much, which matches the
length distribution from Track A — the median issue is far shorter than any of
these thresholds, so the setting only affects the tail.

---
## 2. Hyperparameter search

Cross-validated on the **training split only**. The test split is not consulted
until section 3.

In [4]:
grid = json.loads((TABLES / "grid_search.json").read_text())
for name, rows in grid.items():
    print(f"=== {name} ===")
    display(pd.DataFrame(rows).head(5).reset_index(drop=True))

=== logistic_regression ===


,C,ngram_range,min_df,macro_f1,std,macro_auc
0,10.0,"(1, 2)",2,0.7481,0.0293,0.9020
1,5.0,"(1, 2)",1,0.7472,0.0287,0.9018
2,10.0,"(1, 2)",1,0.7471,0.0252,0.9027
3,25.0,"(1, 2)",1,0.7464,0.0245,0.9036
4,1.0,"(1, 1)",1,0.7463,0.0215,0.8887


=== linear_svm ===


,C,ngram_range,min_df,macro_f1,std,macro_auc
0,1.00,"(1, 2)",1,0.7484,0.0287,None
1,0.50,"(1, 2)",2,0.7466,0.0295,None
2,1.00,"(1, 2)",2,0.7461,0.0304,None
3,0.25,"(1, 2)",2,0.7439,0.0287,None
4,0.50,"(1, 2)",1,0.7435,0.0336,None


Bigrams win consistently, and that is the only setting whose advantage clearly
exceeds the fold-to-fold noise. Everything else is close: the best and fifth-best
configurations differ by about 0.002 macro F1 against a standard deviation of
roughly 0.029.

**An honest note for the report.** The grid was first run under `full` cleaning;
when the ablation moved the pipeline to `light`, the winning `min_df` changed.
Hyperparameters and preprocessing are not independent, and re-tuning after
altering the pipeline is not optional. The two tuned models also disagree —
logistic regression prefers `min_df=2`, the SVM `min_df=1` — which is a
coin-flip within the noise rather than a finding.

---
## 3. Cross-validation on the training split

In [5]:
rows = []
for name, factory in {**CLASSICAL_MODELS, **TUNED_MODELS}.items():
    result = cross_validate(factory, train, k=10, model_name=name)
    rows.append({
        "model": name,
        "macro F1": result.mean_macro_f1,
        "std": result.std_macro_f1,
        "macro AUC": result.mean_macro_auc,
    })
pd.DataFrame(rows).set_index("model").sort_values("macro F1", ascending=False).round(4)

,macro F1,std,macro AUC
model,,,
TF-IDF + Linear SVM (tuned),0.7499,0.0430,NaN
TF-IDF + Logistic Regression,0.7478,0.0409,0.9046
TF-IDF + Logistic Regression (tuned),0.7446,0.0398,0.9036
TF-IDF + Linear SVM,0.7426,0.0367,NaN
TF-IDF + Random Forest,0.7252,0.0316,0.8904
TF-IDF + Naive Bayes,0.6919,0.0334,0.8622
TF-IDF + Complement NB,0.6902,0.0311,0.8582


`LinearSVC` has no probability head, so its AUC is `NaN`. That is deliberate:
wrapping it in probability calibration would change the model being measured.
The harness reports `None` rather than approximating an AUC from hard labels.

Random forest trails the linear models, which is expected — trees cope poorly
with the very high-dimensional sparse features TF-IDF produces. It is a finding,
not a bug.

---
## 4. Final results on the competition protocol

In [6]:
board = leaderboard_from_disk(TABLES)
classical = board[board.index.str.contains("TF-IDF|SetFit")]
display(classical)
print(f"best classical: {classical['overall'].iloc[1]:.4f}  "
      f"(baseline {SETFIT_OVERALL})")

,react,tensorflow,vscode,bitcoin,opencv,overall,AUC,vs SetFit
model,,,,,,,,
SetFit (NLBSE'24 baseline),0.8718,0.8644,0.8262,0.7555,0.8173,0.8270,NaN,0.0000
Ensemble (SetFit + TF-IDF + MPNet),0.8505,0.8520,0.7957,0.7691,0.8168,0.8168,0.9319,-0.0102
SetFit (MPNet),0.8438,0.8710,0.8104,0.7488,0.7771,0.8102,0.9190,-0.0168
Ensemble (SetFit + TF-IDF),0.8396,0.8521,0.7922,0.7459,0.7966,0.8053,0.9244,-0.0217
SetFit (reproduction),0.8322,0.8414,0.7816,0.7464,0.7896,0.7982,0.9181,-0.0288
Ensemble (TF-IDF + MPNet + CNN),0.8296,0.8395,0.7494,0.7694,0.7665,0.7909,0.9245,-0.0361
Ensemble (TF-IDF + MPNet),0.8147,0.7856,0.7762,0.7497,0.7925,0.7838,0.9198,-0.0432
"SetFit (MiniLM, matched settings)",0.7934,0.8618,0.7624,0.7170,0.7735,0.7816,0.9144,-0.0454
TF-IDF + Logistic Regression (tuned),0.8334,0.8155,0.7234,0.6793,0.7496,0.7603,0.9018,-0.0667


best classical: 0.8168  (baseline 0.827)


The best classical configuration reaches **0.7603**, leaving a gap of 0.0667 to
the SetFit baseline.

Note that `bitcoin/bitcoin` and `microsoft/vscode` are hard for every model,
which mirrors the baseline's own per-repository spread — SetFit also scores
lowest on `bitcoin/bitcoin` (0.7555). Project difficulty is a property of the
data, not of any one approach.